# ÍSKOS 2024 — 03. Spáir líkanið rétt fyrir þér?

Fylltu inn þín eigin svör í `my_answers` hér að neðan og keyrðu vinnubókina. Slembiskógurinn úr 02-vinnubókinni spáir hvern þú kýst, og **SHAP-fossamynd** sýnir hvaða svör ýttu líkaninu í þá átt. Að lokum sjáum við **heildaryfirlit** yfir hvaða eiginleikar bera mest forspárgildi um þann flokk fyrir allt prufuúrtakið.

Það er ágætlega líklegt að líkanið spái rangt fyrir um þig. Líkanið hefur rétt fyrir sér um tæplega annan hvern svaranda (sjá nákvæmnina í 02-vinnubókinni). Sjáðu samt **hvers vegna** það spáði eins og það gerði.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

import iskos_style as ik   # ÍSKOS litir + þema (iskos_style.py) — sami stíll og rsetup.R

PROJECT = Path.cwd().parent
DATA = PROJECT / "data"
MODELS = PROJECT / "models"
FIGS = PROJECT / "figures"; FIGS.mkdir(exist_ok=True)

ik.apply_rc()   # ÍSKOS bakgrunnur + letur fyrir allar myndir

# Stilla SHAP-litina á iskos_color_main2 (rauður = auknar líkur, blár = lægri) — Mynd 4.
# shap 0.51 notar `style`-kerfi; við skrifum yfir aðal- og auka-litina fyrir fossamyndina.
from shap.plots._style import set_style as _shap_set_style
_shap_set_style(
    primary_color_positive=ik.ISKOS_MAIN2[0],     # "#cc3e2a"
    primary_color_negative=ik.ISKOS_MAIN2[1],     # "#3d6dba"
    secondary_color_positive="#e08a7e",           # ljósari rauður (skekkjustrik)
    secondary_color_negative="#8fb0dd",           # ljósari blár
)

df = pd.read_csv(DATA / "iskos_clean.csv")
# Index meta by the Icelandic question text — same as df's column names + my_answers keys.
meta = pd.read_csv(DATA / "feature_meta.csv").set_index("label")
rf = joblib.load(MODELS / "rf.pkl")
feature_columns = json.load(open(MODELS / "feature_columns.json"))

print(f"Líkan: {type(rf).__name__} með {rf.n_estimators} trjám, {rf.n_classes_} flokka")
print(f"Eiginleikar (eftir one-hot): {len(feature_columns)}")

# Short-label lookup for SHAP axis text (Mynd 3 & Mynd 4).
# When `short_label` in feature_meta.csv is non-empty, use it instead of the full Icelandic question.
# This keeps plot rows readable without changing model column names. See response_v2.md for
# the translation prompt that produced the short forms.
def short_label(label: str) -> str:
    """Return the short version of an Icelandic question if available, else the full label."""
    if label in meta.index:
        s = meta.loc[label, "short_label"] if "short_label" in meta.columns else ""
        if isinstance(s, str) and s.strip():
            return s.strip()
    return label

def rename_for_plot(columns):
    """Translate one-hot column names (e.g. 'Kjördæmi_Suðvesturkjördæmi') back through short_label.

    Falls back to the original column name when no mapping applies."""
    out = []
    for col in columns:
        # Exact match — numeric / ordinal variables keep their full label as-is.
        if col in meta.index:
            out.append(short_label(col))
            continue
        # One-hot pattern: split on the LAST "_" — but original labels can contain "_" too,
        # so try progressively shorter prefixes.
        matched = None
        for lbl in meta.index:
            if col.startswith(lbl + "_"):
                value = col[len(lbl) + 1:]
                matched = f"{short_label(lbl)} = {value}"
                break
        out.append(matched if matched else col)
    return out

# Value-label lookup for SHAP waterfall (Mynd 4): convert numeric ordinal answers (e.g. 4) into
# their Icelandic text ("frekar óæskilegt") at plot time. Falls back silently when no mapping exists.
import json as _json
_value_labels_path = DATA / "value_labels.json"
value_labels = _json.loads(_value_labels_path.read_text(encoding="utf-8")) if _value_labels_path.exists() else {}

def humanize_data_for_plot(my_row_df):
    """Return a copy of my_row with ordinal codes replaced by their Icelandic text labels.
    Strings (nominal columns) are kept as-is."""
    out = my_row_df.copy()
    for col in out.columns:
        mapping = value_labels.get(col)
        if not mapping:
            continue
        out[col] = out[col].astype(str).map(lambda v: mapping.get(str(int(float(v))), v) if v not in ("", "nan") else v)
    return out


## 1. Yfirlit yfir þær spurningar sem líkanið notar

Líkanið notar eiginleika úr hópum 1, 2, 3, 6, 7 og 8 (sjáðu `feature_meta.csv`). Hér eru þeir allir með íslenskri lýsingu og leyfilegum gildum. Notaðu þetta sem uppflettilista þegar þú fyllir inn í `my_answers` fyrir neðan.

In [ ]:
LEAK_GROUPS = {"4 – Málefnaeign", "5 – Flokkshollusta og leiðtogar"}
EXCLUDE_VARS = set()   # same exclusions as in 02_models
used_questions = [
    q for q in df.columns
    if q != "flokkur" and q in meta.index
    and meta.loc[q, "group"] not in LEAK_GROUPS
    and meta.loc[q, "short_name"] not in EXCLUDE_VARS
]

current_group = None
for q in used_questions:
    info = meta.loc[q]
    if info["group"] != current_group:
        current_group = info["group"]
        print(f"\n## {current_group}")
    if info["dtype"] == "nominal":
        opts = sorted(df[q].dropna().unique().tolist())
        opt_str = " | ".join(repr(o) for o in opts)
        print(f"  [{info['short_name']}] {q}")
        print(f"           ↳ einn af: {opt_str}")
    else:
        valid = df[q][df[q] != -1]
        lo, hi = (int(valid.min()), int(valid.max())) if len(valid) else (-1, -1)
        print(f"  [{info['short_name']}] {q}")
        print(f"           ↳ heiltala {lo}..{hi}   (-1 = ekki svarað)")

## 2. Skrifaðu þín svör í `my_answers`

Sjálfgefin gildi eru **algengasta** flokkagildi (mode) fyrir flokkabreytur og **miðgildi** (median) fyrir raðbreytur — m.ö.o. „dæmigerður ÍsKos-svarandi“. Breyttu þeim sem eiga ekki við þig. Lyklar verða að passa nákvæmlega við breytuheitin hér að ofan.

In [ ]:
my_answers = {
    # ---------- 1 – Félagsleg staða ----------
    "Kjördæmi":                                              "Suðvesturkjördæmi",
    "Aldur svaranda":                                         52,
    "Kyn svaranda (karl/kona/kvár)":                          "Kona",
    "Menntun svaranda (flokkuð)":                             "MA, MSc eða sambærilegt",
    "Búseta í eigin húsnæði, leigu eða annars":               "Bý í eigin húsnæði",
    "Launþegi eða sjálfstæður atvinnurekandi":                "Launþegi",
    "Starfar hjá hinu opinbera eða einkarekstri":             "Í einkageira",
    "Meðlimur í stéttafélagi":                                1,
    "Hjúskaparstaða":                                         "Í hjónabandi",
    "Fjöldi heimilismanna":                                   2,
    "Fjöldi barna 17 ára og yngri á heimilinu":               0,
    # ---------- 2 – Málefnaviðhorf  (oftast 1=mjög sammála ... 5=mjög ósammála) ----------
    "Æskilegt að Ísland gangi í ESB":                         4,
    "Umhverfisaðgerðir á móti atvinnusköpun":                 3,
    "Ríkið eigi að jafna dreifingu tekna":                    2,
    "Skatta ætti að minnka þrátt fyrir niðurskurð":           4,
    "Draga úr viðgangi höfuðborgar til að auka velmegun á landsbyggð": 3,
    "Öll atkvæði eigi að gilda jafnt við úthlutun þingsæta":  2,
    "Stefnt að auknum einkarekstri í heilbrigðisþjónustu":    3,
    "Innflytjendur séu ógn við þjóðareinkenni":               4,
    "EES-samningurinn ógnar fullveldi Íslands":               4,
    "Barátta femínista gegn kynbundnu ofbeldi gengið of langt": 4,
    "Viðskiptabankar eigi að vera í einkaeigu":               4,
    "Innflytjendur eigi jafnan rétt til félagslegs húsnæðis": 2,
    "Ríkisstjórn eigi að leggja áherslu á orkufrek? stóriðju": 4,
    "Rýmka innflutning á landbúnaðarafurðum":                 3,
    "Afglæpavæðing neysluskammta fíkniefna":                  2,
    "Jafnrétti kynja gengið of langt":                        3,
    # ---------- 3 – Frammistaða stjórnvalda ----------
    "Frammistaða ríkisstjórnar (VG–D–B) síðustu þrjú ár":     3,
    "Efnahagsástand á Íslandi síðustu 12 mánuði":             3,
    # ---------- 6 – Viðhorf og hegðun ----------
    "Áhugi á stjórnmálum":                                    3,
    "Traust til stjórnmálafólks":                             3,
    "Ánægja með lýðræðið á Íslandi":                          2,
    "Kjósa er samfélagsleg skylda eða frjálst val":           4,
    "Útbreiðsla spillingar meðal stjórnmálamanna":            2,
    "Meðlimur í stjórnmálaflokki":                            2,
    "Hvenær var endanleg ákvörðun um flokksval tekin":        3,
    # ---------- 7 – Miðlar ----------
    "Tími varið í fréttir í kosningabaráttunni":              3,
    "Miðill notaður mest til frétta":                         "Fréttasíður á netinu",
    # ---------- 8 – Hægri/Vinstri ----------
    "Eigin staðsetning á vinstri-hægri kvarða":               5,
}
missing = set(used_questions) - set(my_answers)
extra   = set(my_answers) - set(used_questions)
assert not missing and not extra, (
    f"my_answers ósamræmi:\n  vantar: {missing}\n  of mikið: {extra}"
)
print("my_answers eru tilbúin —", len(my_answers), "breytur.")

## 3. Skoðum hvernig gögnin líta út eftir one-hot kóðun

Við notum sömu kóðun á gögnunum og notuð var í 02-vinnubókinni: `pd.get_dummies` snertir strengjadálka, raðbreytur fara í gegn óbreyttar. Síðan **endurröðum** við dálkunum við þann lista sem var vistaður með líkaninu (`feature_columns`) — þetta er nauðsynlegt, því líkanið man röð eiginleikanna sem það lærði á.

In [ ]:
my_row_raw = pd.DataFrame([my_answers])
# Apply the same one-hot pipeline used in NB02, then reindex to the exact column order.
my_row = pd.get_dummies(my_row_raw).reindex(columns=feature_columns, fill_value=0)
print(f"Áður: {my_row_raw.shape}  →  Eftir one-hot + reindex: {my_row.shape}")
# Show only the columns that turned on for this row (one-hot 1's + non-zero ordinals)
nonzero = my_row.iloc[0][my_row.iloc[0] != 0]
print("\nGildi sem eru ekki 0 í þinni röð:")
print(nonzero.to_string())

## 4. Spá: líkanið gefur líkur á hverjum flokki

In [ ]:
probs = rf.predict_proba(my_row)[0]
prob_s = pd.Series(probs, index=rf.classes_).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
# Litur hverrar súlu = flokkslitur (party_colors via class_colors).
ax.barh(prob_s.index, prob_s.values, color=ik.class_colors(prob_s.index))
for y_pos, v in enumerate(prob_s.values):
    ax.text(v + 0.005, y_pos, f"{v:.2f}", va="center", fontsize=9, color=ik.ISKOS_TEXT_MID)
ax.set_xlabel("Líkur")
ax.set_xlim(0, max(prob_s.values) * 1.18)
ik.style_iskos(ax, title="Spá slembiskógarins fyrir þig")
plt.tight_layout()
plt.show()

top = prob_s.idxmax()
print(f"\nLíklegasti flokkur: {top}   (líkur: {prob_s[top]:.2f})")

## 5. SHAP útskýring fyrir þína spá

**Fossagraf** (e. waterfall) sýnir hvernig hvert svar þitt **ýtti líkaninu í átt að** spáða flokknum (rauðar súlur = auknar líkur) eða **frá honum** (bláar súlur = lægri líkur). Byrjað er á væntigildinu `E[f(x)]` (sem er hér meðaltal líkana yfir prufuúrtakið) og endað á þeim líkum sem þitt svar gefur. Því stærri sem gildi er, því meira vegur svarið.

In [ ]:
import re
explainer = shap.TreeExplainer(rf)
# SHAP returns a 3-D array (n_samples, n_features, n_classes) for multi-class trees in v0.46+.
shap_vals = explainer.shap_values(my_row)
expected = explainer.expected_value
cls_idx = list(rf.classes_).index(top)

# Normalise shap_vals shape across shap versions.
if isinstance(shap_vals, list):
    sv_top = shap_vals[cls_idx][0]
    base = expected[cls_idx]
elif shap_vals.ndim == 3:
    sv_top = shap_vals[0, :, cls_idx]
    base = expected[cls_idx]
else:
    sv_top = shap_vals[0]
    base = expected

# Convert ordinal-column values from integer codes to Icelandic text labels so the waterfall
# reads "Æskilegt að Ísland gangi í ESB = frekar óæskilegt" instead of "= 4".
my_row_display = humanize_data_for_plot(my_row)
exp = shap.Explanation(
    values=sv_top,
    base_values=base,
    data=my_row_display.iloc[0].values,
    feature_names=rename_for_plot(feature_columns),
)
# Render with show=False so we can post-process the y-tick labels:
#  1. Localise the auto-generated "X other features" line to Icelandic.
#  2. Drop the redundant "True = " prefix from one-hot rows — the feature name
#     already includes the chosen value (e.g. "Helsti fréttamiðill = Fréttasíður á netinu").
# Litir súlnanna (rauður/blár) eru stilltir á iskos_color_main2 í fyrsta kóðareit (set_style).
shap.plots.waterfall(exp, max_display=11, show=False)   # 10 largest contributions + the rest
fig = plt.gcf(); ax = plt.gca()
fig.set_size_inches(9, 5.6)                             # book figure size (Mynd 8.4)
fig.set_facecolor(ik.ISKOS_BG); ax.set_facecolor(ik.ISKOS_BG)   # ÍSKOS bakgrunnur (theme_iskos)
def _clean_ytick(text: str) -> str:
    text = re.sub(r"(\d+) other features?", lambda m: f"{m.group(1)} aðrar breytur", text)
    text = re.sub(r"^True = ", "", text)
    return text
ax.set_yticklabels([_clean_ytick(lbl.get_text()) for lbl in ax.get_yticklabels()])
# Larger, consistent text for print: tick labels, value labels and the E[f(X)] / f(x) annotations.
for lbl in ax.get_yticklabels():
    lbl.set_fontsize(11)
for txt in fig.findobj(match=lambda o: hasattr(o, 'set_fontsize') and o.__class__.__name__ == 'Text'):
    if txt.get_text() and txt not in ax.get_yticklabels():
        txt.set_fontsize(max(txt.get_fontsize(), 10.5))
fig.savefig(FIGS / 'mynd_8_4_fossarit.png', dpi=300, bbox_inches='tight', facecolor=ik.ISKOS_BG)
fig.savefig(FIGS / 'mynd_8_4_fossarit.svg', bbox_inches='tight', facecolor=ik.ISKOS_BG)
plt.show()

## 6. Heildaryfirlit: hvaða eiginleikar bera mest forspárgildi yfir allt prufuúrtakið?

**SHAP summary plot** keyrt á öllum gögnunum (1012 svarendum) sýnir hvaða eiginleikar bera mest forspárgildi um þann flokk sem þú færðu sem efstu spá. Hver punktur er einn svarandi: lóðrétt staðsetning sýnir eiginleikann, lárétt staðsetning sýnir SHAP-gildið (jákvætt = ýtir í átt að flokknum), og **liturinn** sýnir hvort gildi eiginleikans hjá þeim svaranda var hátt eða lágt.

In [ ]:
# Encode the entire dataset using the same pipeline as NB02.
leak_questions = meta.index[meta["group"].isin(LEAK_GROUPS)].tolist()
X_full = pd.get_dummies(
    df.drop(columns=["flokkur"] + leak_questions)
).reindex(columns=feature_columns, fill_value=0)

sv_all = explainer.shap_values(X_full)

if isinstance(sv_all, list):
    sv_all_top = sv_all[cls_idx]
elif sv_all.ndim == 3:
    sv_all_top = sv_all[:, :, cls_idx]
else:
    sv_all_top = sv_all

X_full_plot = X_full.rename(columns=dict(zip(X_full.columns, rename_for_plot(X_full.columns))))
shap.summary_plot(
    sv_all_top, X_full_plot, max_display=15, show=False,
    plot_size=(9, 6),
)
fig = plt.gcf()
fig.set_facecolor(ik.ISKOS_BG)                       # ÍSKOS bakgrunnur (theme_iskos)
for _ax in fig.get_axes():
    _ax.set_facecolor(ik.ISKOS_BG)
plt.suptitle(f"SHAP-yfirlit fyrir spáðan flokk: {top}", y=1.02)
plt.tight_layout()
plt.savefig(FIGS / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()

---

**Prufaðu sjálf/ur/t:** skrunaðu upp, breyttu einhverjum gildum í `my_answers` og keyrðu kóðan aftur (`Cell → Run All Below` í Jupyter). Sjáðu hvernig spáin og fossagrafið breytast.